In [1]:
# Google Colab drive mount disabled for local execution
print('Running locally. Using local relative paths.')


Running locally. Using local relative paths.


In [2]:
import numpy as np
corn = np.load("../corn4_data.npz")
soy = np.load("../soybean_data.npz")
wheat = np.load("../wheat_data.npz")

In [3]:
X_corn, X_extra_corn, y_corn = corn["X"], corn["X_extra"], corn["y"]
fips_corn, year_corn = corn["fips"], corn["year"]

X_soy, X_extra_soy, y_soy = soy["X"], soy["X_extra"], soy["y"]
fips_soy, year_soy = soy["fips"], soy["year"]

X_wheat, X_extra_wheat, y_wheat = wheat["X"], wheat["X_extra"], wheat["y"]
fips_wheat, year_wheat = wheat["fips"], wheat["year"]

In [4]:
CROP_MAP = {"corn": 0, "soybean": 1, "wheat": 2}

crop_corn = np.full(len(y_corn), CROP_MAP["corn"])
crop_soy = np.full(len(y_soy), CROP_MAP["soybean"])
crop_wheat = np.full(len(y_wheat), CROP_MAP["wheat"])

In [5]:
X_all = np.concatenate([X_corn, X_soy, X_wheat])
X_extra_all = np.concatenate([X_extra_corn, X_extra_soy, X_extra_wheat])
y_all = np.concatenate([y_corn, y_soy, y_wheat])

fips_all = np.concatenate([fips_corn, fips_soy, fips_wheat])
year_all = np.concatenate([year_corn, year_soy, year_wheat])
crop_all = np.concatenate([crop_corn, crop_soy, crop_wheat])

In [6]:
print(X_all.shape)
print(X_extra_all.shape)
print(y_all.shape)

(19873, 6, 8)
(19873, 8)
(19873,)


In [7]:
print("Corn:", X_corn.shape, X_extra_corn.shape, y_corn.shape)
print("Soy:", X_soy.shape, X_extra_soy.shape, y_soy.shape)
print("Wheat:", X_wheat.shape, X_extra_wheat.shape, y_wheat.shape)

Corn: (8375, 6, 8) (8375, 8) (8375,)
Soy: (7449, 6, 8) (7449, 8) (7449,)
Wheat: (4049, 6, 8) (4049, 8) (4049,)


In [8]:
from sklearn.preprocessing import StandardScaler

# FIPS standardization across all crops
fips_corn_str = np.array([str(int(float(f))).zfill(5) if isinstance(f, (int, float, np.number)) else str(f).zfill(5) for f in fips_corn])
fips_soy_str = np.array([str(int(float(f))).zfill(5) if isinstance(f, (int, float, np.number)) else str(f).zfill(5) for f in fips_soy])
fips_wheat_str = np.array([str(int(float(f))).zfill(5) if isinstance(f, (int, float, np.number)) else str(f).zfill(5) for f in fips_wheat])

fips_all_str = np.concatenate([fips_corn_str, fips_soy_str, fips_wheat_str])
all_unique_fips_codes = np.unique(fips_all_str)
fips_to_id = {fips: i for i, fips in enumerate(all_unique_fips_codes)}

# Lists to store scaled training and test sets
scalers = {}  # Store target scalers per crop
X_train_list, X_test_list = [], []
X_lag_train_list, X_lag_test_list = [], []
y_train_scaled_list, y_test_scaled_list = [], []
y_test_raw_list = []
fips_train_list, fips_test_list = [], []
crop_train_list, crop_test_list = [], []

# Cast years
year_corn_int = year_corn.astype(float).astype(int)
year_soy_int = year_soy.astype(float).astype(int)
year_wheat_int = year_wheat.astype(float).astype(int)

# Loop and scale each crop individually
for crop_name, (X, X_extra, y, year, fips, crop_val) in {
    "corn": (X_corn, X_extra_corn, y_corn, year_corn_int, fips_corn_str, CROP_MAP["corn"]),
    "soybean": (X_soy, X_extra_soy, y_soy, year_soy_int, fips_soy_str, CROP_MAP["soybean"]),
    "wheat": (X_wheat, X_extra_wheat, y_wheat, year_wheat_int, fips_wheat_str, CROP_MAP["wheat"])
}.items():
    
    # Create train and test masks
    train_mask = year < 2022
    test_mask = year == 2022
    
    # Scale sequences (X)
    seq_scaler = StandardScaler()
    X_train_c = X[train_mask]
    X_test_c = X[test_mask]
    X_train_c_scaled = seq_scaler.fit_transform(X_train_c.reshape(-1, X_train_c.shape[-1])).reshape(X_train_c.shape)
    X_test_c_scaled = seq_scaler.transform(X_test_c.reshape(-1, X_test_c.shape[-1])).reshape(X_test_c.shape)
    
    # Scale lag / extra features (X_extra)
    lag_scaler = StandardScaler()
    X_lag_train_c = X_extra[train_mask]
    X_lag_test_c = X_extra[test_mask]
    X_lag_train_c_scaled = lag_scaler.fit_transform(X_lag_train_c)
    X_lag_test_c_scaled = lag_scaler.transform(X_lag_test_c)
    
    # Scale target yields (y) - PER CROP to prevent variance imbalance!
    y_scaler = StandardScaler()
    y_train_c = y[train_mask]
    y_test_c = y[test_mask]
    y_train_c_scaled = y_scaler.fit_transform(y_train_c.reshape(-1, 1)).flatten()
    y_test_c_scaled = y_scaler.transform(y_test_c.reshape(-1, 1)).flatten()
    
    # Store the target scaler for later inverse transform
    scalers[crop_val] = y_scaler
    
    # Encode FIPS
    fips_train_c_encoded = np.array([fips_to_id[f] for f in fips[train_mask]])
    fips_test_c_encoded = np.array([fips_to_id[f] for f in fips[test_mask]])
    
    # Append
    X_train_list.append(X_train_c_scaled)
    X_test_list.append(X_test_c_scaled)
    X_lag_train_list.append(X_lag_train_c_scaled)
    X_lag_test_list.append(X_lag_test_c_scaled)
    y_train_scaled_list.append(y_train_c_scaled)
    y_test_scaled_list.append(y_test_c_scaled)
    y_test_raw_list.append(y_test_c)
    fips_train_list.append(fips_train_c_encoded)
    fips_test_list.append(fips_test_c_encoded)
    crop_train_list.append(np.full(len(y_train_c), crop_val))
    crop_test_list.append(np.full(len(y_test_c), crop_val))

# Concatenate all scaled datasets
X_train_scaled = np.concatenate(X_train_list)
X_test_scaled = np.concatenate(X_test_list)
X_lag_train_scaled = np.concatenate(X_lag_train_list)
X_lag_test_scaled = np.concatenate(X_lag_test_list)
y_train_scaled = np.concatenate(y_train_scaled_list)
y_test_scaled = np.concatenate(y_test_scaled_list)
y_test_raw = np.concatenate(y_test_raw_list)
fips_train_encoded = np.concatenate(fips_train_list)
fips_test_encoded = np.concatenate(fips_test_list)
crop_train = np.concatenate(crop_train_list)
crop_test = np.concatenate(crop_test_list)

print("Data splitting and per-crop scaling complete.")


Data splitting and per-crop scaling complete.


In [9]:
# Sequence scaling was performed crop-by-crop in Cell 7.
print("Sequence training shape:", X_train_scaled.shape)
print("Sequence testing shape:", X_test_scaled.shape)


Sequence training shape: (16303, 6, 8)
Sequence testing shape: (3570, 6, 8)


In [10]:
# Extra features / lag scaling was performed crop-by-crop in Cell 7.
print("Lag training shape:", X_lag_train_scaled.shape)
print("Lag testing shape:", X_lag_test_scaled.shape)


Lag training shape: (16303, 8)
Lag testing shape: (3570, 8)


In [11]:
# Target scaling was performed crop-by-crop in Cell 7 to prevent variance imbalance.
print("Target training shape:", y_train_scaled.shape)
print("Target testing shape:", y_test_scaled.shape)


Target training shape: (16303,)
Target testing shape: (3570,)


In [12]:
print("Final Concatenated Shape verification:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_lag_train_scaled:", X_lag_train_scaled.shape)
print("y_train_scaled:", y_train_scaled.shape)


Final Concatenated Shape verification:
X_train_scaled: (16303, 6, 8)
X_lag_train_scaled: (16303, 8)
y_train_scaled: (16303,)


In [13]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate, Dropout, Embedding, Flatten
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# sequence input
seq_input = Input(shape=(6, 8))

x = LSTM(128, return_sequences=True)(seq_input)
x = tf.keras.layers.LayerNormalization()(x)
x = LSTM(64)(x)
x = Dropout(0.3)(x)
x = Dense(32, activation='relu')(x)

# lag input
lag_input = Input(shape=(8,))
y_lag = Dense(16, activation='relu')(lag_input)

# fips embedding
fips_input = Input(shape=(1,))
fips_emb = Embedding(input_dim=3000, output_dim=8)(fips_input)
fips_emb = Flatten()(fips_emb)

# crop embedding 🔥
crop_input = Input(shape=(1,))
crop_emb = Embedding(input_dim=3, output_dim=4)(crop_input)
crop_emb = Flatten()(crop_emb)

# combine
combined = Concatenate()([x, y_lag, fips_emb, crop_emb])

z = Dense(64, activation='relu')(combined)
z = Dropout(0.3)(z)
z = Dense(32, activation='relu')(z)

output = Dense(1)(z)

model = Model(
    inputs=[seq_input, lag_input, fips_input, crop_input],
    outputs=output
)

model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='mse',
    metrics=['mae']
)

model.summary()

I0000 00:00:1781195395.894793   20275 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781195395.896600   20275 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781195395.987347   20275 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1781195398.437897   20275 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781195398.440803   20275 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


E0000 00:00:1781195398.941755   20275 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6, 8)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 6, 128)    │     70,144 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 6, 128)    │        256 │ lstm[0][0]        │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 64)        │     49,408 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 8)      │     24,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 4)      │         12 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      2,080 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        144 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8)         │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 4)         │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 60)        │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_1[0][0],    │
│                     │                   │            │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      3,904 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 1)         │         33 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 152,061 (593.99 KB)

 Trainable params: 152,061 (593.99 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    [X_train_scaled, X_lag_train_scaled, fips_train_encoded, crop_train],
    y_train_scaled,
    validation_data=([X_test_scaled, X_lag_test_scaled, fips_test_encoded, crop_test], y_test_scaled),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    shuffle=False
)


Epoch 1/50


  1/255 ━━━━━━━━━━━━━━━━━━━━ 28:05 7s/step - loss: 1.1945 - mae: 0.9097

  4/255 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - loss: 1.0828 - mae: 0.8707 

  7/255 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 1.0033 - mae: 0.8393

 10/255 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.9772 - mae: 0.8212

...

247/255 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.1715 - mae: 0.3168

249/255 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.1715 - mae: 0.3169

252/255 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.1716 - mae: 0.3169

255/255 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.1717 - mae: 0.3170

255/255 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 0.1796 - mae: 0.3245 - val_loss: 0.4793 - val_mae: 0.5508


In [15]:
from sklearn.metrics import r2_score

# Generate predictions
y_pred_scaled = model.predict([X_test_scaled, X_lag_test_scaled, fips_test_encoded, crop_test]).flatten()

y_test_by_crop = {}
y_pred_by_crop = {}

# Evaluate each crop crop-by-crop using its own inverse scaler
for crop_name, crop_val in CROP_MAP.items():
    mask = (crop_test == crop_val)
    
    y_true_c = y_test_raw[mask]
    y_pred_c_scaled = y_pred_scaled[mask]
    
    y_scaler = scalers[crop_val]
    y_pred_c = y_scaler.inverse_transform(y_pred_c_scaled.reshape(-1, 1)).flatten()
    
    y_test_by_crop[crop_val] = y_true_c
    y_pred_by_crop[crop_val] = y_pred_c
    
    r2_crop = r2_score(y_true_c, y_pred_c)
    print(f"🌽 {crop_name.capitalize()} R² inside Multi-Crop: {r2_crop:.6f}")

# Calculate global combined R2 on raw values
all_y_true = np.concatenate([y_test_by_crop[0], y_test_by_crop[1], y_test_by_crop[2]])
all_y_pred = np.concatenate([y_pred_by_crop[0], y_pred_by_crop[1], y_pred_by_crop[2]])
global_r2_raw = r2_score(all_y_true, all_y_pred)
print(f"\n🔥 Multi-Crop Combined R² (on raw yields): {global_r2_raw:.6f}")


  1/112 ━━━━━━━━━━━━━━━━━━━━ 1:00 549ms/step

  8/112 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step    

 16/112 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step

 24/112 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step

 32/112 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step

...

 98/112 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

105/112 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step

112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step

112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step


🌽 Corn R² inside Multi-Crop: 0.651352
🌽 Soybean R² inside Multi-Crop: 0.702758
🌽 Wheat R² inside Multi-Crop: 0.631352

🔥 Multi-Crop Combined R² (on raw yields): 0.904243
